# 💬 AIRA Assistant: Chatbot Hybrid (LLM + Semi RAG) & Guardrails (MVP)

Selamat datang di **Hands-on 2** proyek AIRA! Notebook ini dirancang untuk membangun asisten interaktif cerdas yang mampu menjawab pertanyaan umum (*general*) menggunakan kemampuan LLM, sekaligus mencari informasi spesifik dari dokumen referensi yang Anda unggah (*Semi RAG*), dengan tetap mematuhi batasan (*guardrails*) yang ketat dari PRD.

### Alur Kerja RAG (Workflow):
1. **Unggah Dokumen Resmi**: Memasukkan file PDF/TXT panduan WASH atau FAQ kekeringan ke Colab.
2. **Ekstraksi Teks**: Membaca isi dokumen halaman demi halaman beserta metadata (judul, halaman, sumber).
3. **Chunking & Overlap**: Memecah teks dokumen besar menjadi potongan-potongan kecil (*chunks*) agar pencarian lebih akurat.
4. **Embedding & ChromaDB**: Mengubah teks menjadi vektor numerik menggunakan `Sentence Transformers` dan menyimpannya ke database vektor lokal `ChromaDB`.
5. **Retrieval & Generation**: Mencari chunk paling relevan berdasarkan pertanyaan pengguna, lalu menyusun jawaban empatik berbasis bukti menggunakan model Gemini.

## 📋 0. Dokumentasi Assignment (🆕 TAMBAHAN)

> Bagian ini melengkapi komponen **Problem Statement, Feature Description, Input & Data Source** pada assignment.

### 🎯 Problem Statement
Perempuan dan keluarga di daerah rawan kekeringan, khususnya ibu rumah tangga yang mengelola kebutuhan air keluarga, kesulitan menemukan panduan pengelolaan air, sanitasi, dan Manajemen Kebersihan Menstruasi (MHM) yang sederhana, tepat, dan tidak menghakimi, terutama saat cadangan air mulai menipis. Informasi yang ada tersebar di banyak dokumen dan sering terlalu teknis.

**Pengguna sasaran:** perempuan dan keluarga di daerah rawan kekeringan (persona: Ibu Siti Rahma, Gunungkidul).

### 🧩 Feature Description
**AIRA Assistant** adalah asisten AI untuk menjawab pertanyaan, menjelaskan hasil Kalkulator Air, dan memberi panduan air, sanitasi, MHM, serta penanganan krisis air. Fitur ini adalah salah satu dari tiga fitur utama AIRA (bersama Kalkulator Air dan Navigasi Air).

Pendekatan **Hybrid (Semi-RAG)**: dokumen referensi resmi dicari lewat vector search (ChromaDB) lalu dipakai sebagai konteks oleh LLM (Gemini). Jika tidak ada dokumen yang cukup relevan, asisten menjawab dengan pengetahuan umum dan menyatakannya secara transparan.

**Batasan dari PRD yang diuji di notebook ini:**
1. AI tidak boleh mengubah angka atau status hasil kalkulator.
2. AI tidak boleh mengarang ketersediaan titik air, jam operasional, atau kualitas air.
3. Jika layanan AI gagal, pengguna tetap mendapat pesan yang jelas dan fitur lain tetap dapat dipakai.
4. AI tidak boleh mengganti status "Belum Dapat Ditentukan" menjadi Aman, dan harus menjelaskan keterbatasan informasi bila data belum cukup.

### 📥 Input & Data Source
| Input | Keterangan | Sumber |
|---|---|---|
| Pertanyaan pengguna | Teks bahasa Indonesia | Diketik pengguna (di notebook: daftar kasus uji) |
| Konteks kalkulator | status, alasan status, kondisi sumber, durasi ketahanan, kebutuhan harian (termasuk tambahan MHM), cadangan air, aksi utama, dan rekomendasi rule-based | Fungsi yang **disalin persis** dari notebook Kalkulator Air (bagian 7), sehingga input yang sama menghasilkan status yang sama |
| Dokumen referensi | PDF tentang WASH, MHM, dan ketahanan iklim (UNICEF, WHO, dll.); jumlah yang terunggah dilaporkan otomatis di bagian 2 | Diunggah manual; metadata dan tautan sumber ada di tabel `DOKUMEN_REFERENSI` (bagian 2) |

### 📤 Output yang Diharapkan
Jawaban bahasa Indonesia yang hangat dan praktis, disertai **daftar sumber yang dibuat sistem dari hasil retrieval** (bukan dikarang LLM). Untuk permintaan di luar batasan (mis. mengubah status), jawaban berupa penolakan sopan.

### 🔄 Alur Pipeline
```
PDF referensi → ekstraksi teks → chunking (buang potongan pendek/duplikat) → embedding → ChromaDB
Pertanyaan ─► guard input ─► retrieval (top-k + ambang jarak) ─► prompt (+ konteks kalkulator)
          ─► Gemini (retry bertahap + fallback antar-model) ─► guard output (cek status) ─► jawaban + sumber otomatis
```

### 🗺️ Peta Komponen Assignment → Bagian Notebook
| Komponen | Lokasi |
|---|---|
| Problem Statement, Feature Description, Input & Data Source | Bagian 0 (sel ini) |
| Initial Pipeline | Bagian 1 s.d. 7 |
| Output & Interpretation | Bagian 8 s.d. 11 (termasuk uji logika tanpa API di 9b dan penilaian manual di 9c) |
| Next Steps | Bagian 12 |
| Cara menjalankan (untuk GitHub) | Bagian 13 |

In [ ]:
# 1. Instalasi dependensi utama untuk Vector RAG & Google GenAI SDK
!pip install -q chromadb sentence-transformers pymupdf google-genai requests

import os
import re
import json
import math
import time
import random
import hashlib
import getpass
import pymupdf
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai

print("✅ Instalasi library selesai!")

## 🔑 1. Konfigurasi Google Gemini API Key

Masukkan Google Gemini API Key Anda. Key ini digunakan oleh model generative untuk menyusun jawaban yang grounded dan empatik berdasarkan hasil retrieval dokumen.

In [ ]:
# Mengambil atau meminta Gemini API Key secara aman
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Masukkan Google Gemini API key Anda: ").strip()

# Inisialisasi Google GenAI Client
client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Inisialisasi Google GenAI Client Berhasil!")
print("🤖 Urutan model Gemini (utama → cadangan) ditentukan pada bagian 6.")

## 📁 2. Unggah Dokumen Referensi (PDF / TXT) & Ekstraksi Teks (✏️ REVISI)

Unggah PDF referensi. Metadata resmi tiap dokumen (judul, penerbit, tautan) dicatat di tabel `DOKUMEN_REFERENSI` agar sitasi akurat. Dokumen yang metadatanya belum dilengkapi (nilai `None`) tetap diproses, tetapi sitasinya menampilkan nama file dengan keterangan "judul resmi belum dilengkapi"; lengkapi dari halaman sampul PDF sebelum notebook dikumpulkan.

Sel ini juga melaporkan dokumen yang terdaftar tetapi belum diunggah, serta PDF yang separuh halamannya tidak berteks (kemungkinan hasil scan yang perlu OCR).

In [ ]:
# ✏️ REVISI: menggantikan sel "Unggah Dokumen" yang lama
from google.colab import files
from datetime import datetime, timezone
import pymupdf

# ✏️ ISI / PERIKSA: metadata resmi tiap dokumen. Kunci = potongan nama file (huruf kecil).
# Nilai None = belum diketahui. Lengkapi dari halaman sampul PDF; sebelum diisi, sitasi akan
# menampilkan nama file dan keterangan "judul resmi belum dilengkapi" (bukan teks placeholder).
DOKUMEN_REFERENSI = {
    "kerangka-penguatan-ketahanan-iklim": {
        "title": "Kerangka Penguatan Ketahanan Iklim, Air Minum, Sanitasi, dan Kebersihan",
        "publisher": "UNICEF Indonesia",
        "source_url": "https://www.unicef.org/indonesia/id/wash/laporan/kerangka-penguatan-ketahanan-iklim-air-minum-sanitasi-dan-kebersihan",
    },
    "mhm and child marriage": {
        "title": "MHM dan Pencegahan Pernikahan Anak Usia Dini",
        "publisher": "UNICEF",
        "source_url": "https://www.unicef.org/indonesia/media/9491/file/MHM%20and%20Child%20Marriage%20Prevention%20(Indonesian).pdf",
    },
    "who-ced-phe-wsh-19.149": {
        "title": "Water, sanitation, hygiene and health: a primer for health professionals",
        "publisher": "World Health Organization (WHO), 2019",
        "source_url": "https://www.who.int/publications-detail-redirect/WHO-CED-PHE-WSH-19.149",
    },
    "mhm-comicbook": {                      # ✏️ ISI dari halaman sampul PDF
        "title": None,
        "publisher": None,
        "source_url": None,
    },
    "risk-management-wash": {               # ✏️ ISI dari halaman sampul PDF (Desember 2017)
        "title": None,
        "publisher": None,
        "source_url": None,
    },
}

def normalisasi_nama(filename):
    """'abc (1).pdf' -> 'abc.pdf' (nama ganda akibat upload ulang di Colab)."""
    return re.sub(r"\s*\(\d+\)(?=\.\w+$)", "", filename)

def cari_metadata(filename):
    nama = normalisasi_nama(filename).lower()
    for kunci, meta in DOKUMEN_REFERENSI.items():
        if kunci in nama:
            return dict(meta)
    return None

def lengkapi_metadata(meta, nama_file):
    """Isi nilai kosong dengan keterangan jujur agar sitasi tidak menampilkan teks placeholder."""
    belum_lengkap = any(v in (None, "") for v in meta.values())
    meta = dict(meta)
    meta["title"] = meta.get("title") or f"{nama_file} (judul resmi belum dilengkapi)"
    meta["publisher"] = meta.get("publisher") or "Penerbit belum dilengkapi"
    meta["source_url"] = meta.get("source_url") or "-"
    return meta, belum_lengkap

raw_pages = []
sudah_diproses = set()

print("📂 Silakan unggah file PDF/TXT referensi dari perangkat lokal Anda:")
uploaded = files.upload()

for filename, file_bytes in uploaded.items():
    nama_bersih = normalisasi_nama(filename)
    if not nama_bersih.lower().endswith((".pdf", ".txt")):
        print(f"⚠️ Dilewati (bukan PDF/TXT): {filename}")
        continue
    if nama_bersih in sudah_diproses:
        print(f"⚠️ Dilewati (dokumen ganda): {filename}")
        continue
    sudah_diproses.add(nama_bersih)

    meta = cari_metadata(filename)
    if meta is None:  # dokumen baru yang belum ada di tabel -> isi manual
        print(f"ℹ️ {filename} belum ada di DOKUMEN_REFERENSI, isi manual:")
        meta = {
            "title": input("Judul resmi: ").strip() or None,
            "publisher": input("Penerbit/lembaga resmi: ").strip() or None,
            "source_url": input("URL sumber (opsional): ").strip() or None,
        }
    meta, belum_lengkap = lengkapi_metadata(meta, nama_bersih)
    if belum_lengkap:
        print(f"⚠️ Metadata {nama_bersih} belum lengkap; lengkapi di DOKUMEN_REFERENSI sebelum dipakai sebagai sitasi.")

    print(f"\n📄 Memproses dokumen: {nama_bersih}  →  {meta['title']}")
    waktu = datetime.now(timezone.utc).isoformat()

    if nama_bersih.lower().endswith(".pdf"):
        pdf = pymupdf.open(stream=file_bytes, filetype="pdf")
        n = 0
        for page_num, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if not text:
                continue
            n += 1
            raw_pages.append({
                "filename": nama_bersih, "title": meta["title"], "publisher": meta["publisher"],
                "source_url": meta["source_url"], "page": page_num, "text": text, "ingested_at": waktu,
            })
        print(f"   ✅ {n} halaman teks terekstrak dari {len(pdf)} halaman.")
        if n < len(pdf) / 2:
            print("   ⚠️ Kurang dari separuh halaman berteks; PDF ini mungkin hasil scan/gambar dan perlu OCR.")
    else:
        text_content = file_bytes.decode("utf-8", errors="ignore").strip()
        raw_pages.append({
            "filename": nama_bersih, "title": meta["title"], "publisher": meta["publisher"],
            "source_url": meta["source_url"], "page": 1, "text": text_content, "ingested_at": waktu,
        })
        print("   ✅ Teks berhasil dibaca.")

# Dokumen yang terdaftar tetapi belum diunggah
nama_terunggah = [n.lower() for n in sudah_diproses]
belum_diunggah = [k for k in DOKUMEN_REFERENSI if not any(k in n for n in nama_terunggah)]
print(f"\n📊 Total halaman/dokumen mentah terkumpul: {len(raw_pages)} dari {len(sudah_diproses)} file")
if belum_diunggah:
    print("ℹ️ Terdaftar di DOKUMEN_REFERENSI tetapi belum diunggah:", belum_diunggah)

## 🧩 3. Chunking Dokumen & Pembuatan Metadata Sitasi (✏️ REVISI)

Teks yang panjang dipotong menjadi *chunk* beserta metadata sitasi. Potongan yang sangat pendek (judul, sampul, nomor halaman) dan potongan duplikat dibuang. Pada uji sebelumnya, potongan sampul WHO justru muncul sebagai hasil teratas untuk pertanyaan MHM sehingga bisa tampil sebagai sumber yang tidak mendukung jawaban.

In [ ]:
MIN_CHUNK_CHARS = 200   # potongan yang sangat pendek (judul/sampul/nomor halaman) tidak membawa panduan

def chunk_text(text, chunk_size=1500, overlap=250):
    text = " ".join(text.split())
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        part = text[start:end].strip()
        if part:
            chunks.append(part)
        if end >= len(text):
            break
        start = end - overlap
    return chunks

records = []
dilewati_pendek = dilewati_duplikat = 0
teks_terlihat = set()
for doc in raw_pages:
    for chunk_idx, chunk in enumerate(chunk_text(doc["text"])):
        if len(chunk) < MIN_CHUNK_CHARS:
            dilewati_pendek += 1
            continue
        kunci = hashlib.md5(chunk.lower().encode("utf-8")).hexdigest()
        if kunci in teks_terlihat:          # halaman berulang dengan isi sama
            dilewati_duplikat += 1
            continue
        teks_terlihat.add(kunci)
        records.append({
            "id": f"{doc['filename']}-p{doc['page']}-c{chunk_idx}",
            "text": chunk,
            "metadata": {
                "title": doc["title"][:300],
                "publisher": doc["publisher"][:200],
                "source_url": doc["source_url"][:500],
                "page": int(doc["page"]),
                "chunk_index": int(chunk_idx),
                "filename": doc["filename"][:200]
            }
        })

print(f"🎯 Total chunk yang dihasilkan: {len(records)}")
print(f"   Dilewati: {dilewati_pendek} chunk terlalu pendek (< {MIN_CHUNK_CHARS} karakter), {dilewati_duplikat} chunk duplikat")
if records:
    print("\n📝 Contoh Metadata Chunk Pertama:")
    print(records[0]["metadata"])

## 🔍 4. Embedding & Penyimpanan ke ChromaDB

Kita menggunakan model multibahasa `paraphrase-multilingual-mpnet-base-v2` dari Sentence Transformers untuk mengubah teks chunk menjadi vektor numerik, lalu menyimpannya ke dalam database vektor lokal **ChromaDB**.

In [ ]:
# Inisialisasi model embedding multibahasa yang mendukung Bahasa Indonesia
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
print(f"⏳ Memuat model embedding: {EMBEDDING_MODEL_NAME}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Inisialisasi ChromaDB Persistent Client
CHROMA_PATH = "/content/aira_chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

COLLECTION_NAME = "aira_knowledge_base"
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# Memasukkan data chunk ke ChromaDB secara batch
if records:
    batch_size = 48
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        vectors = embedding_model.encode(
            [r["text"] for r in batch],
            normalize_embeddings=True
        ).tolist()

        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            metadatas=[r["metadata"] for r in batch],
            embeddings=vectors
        )
    print(f"✅ Berhasil mengindeks {collection.count()} chunk ke dalam ChromaDB!")
else:
    print("⚠️ Tidak ada record dokumen untuk diindeks.")

## 🛡️ 5. System Prompt & Guardrails AIRA Assistant (✏️ REVISI)

Guardrails mengikuti batasan PRD: AI tidak mengubah angka atau status, tidak mengarang titik air, jam operasional, atau kualitas air, menjelaskan keterbatasan informasi, dan tidak mengganti status "Belum Dapat Ditentukan" menjadi Aman. Aturan 12 dan 13 baru: penanganan status "Belum Dapat Ditentukan" serta pemisahan jelas antara data kalkulator, panduan dokumen, dan saran umum.

In [ ]:
# ✏️ REVISI: menggantikan sel "System Prompt" yang lama (aturan 1-6 tetap, 7-13 baru)
AIRA_SYSTEM_PROMPT = """
Anda adalah AIRA Assistant, asisten digital yang ramah, hangat, dan peduli untuk membantu keluarga di daerah rawan kekeringan.
Target audiens utama Anda adalah ibu rumah tangga dan masyarakat desa.

ATURAN UTAMA (GUARDRAILS):
1. Jawab menggunakan bahasa Indonesia yang jelas, empatik, membumi, dan mudah dipahami.
2. Jika ada Dokumen Referensi (Semi RAG) yang dilampirkan di bawah, gunakan informasi tersebut untuk menjawab pertanyaan spesifik. Jika pertanyaan bersifat umum, jawablah dengan pengetahuan umum yang relevan dan aman.
3. Jika pengguna memberikan data hasil kalkulator, bantu jelaskan maknanya secara sederhana dan gunakan angka PERSIS seperti yang tertera.
4. JANGAN PERNAH menghitung ulang, mengubah angka hasil perhitungan, atau mengubah status kondisi air (Aman, Waspada, Mode Krisis, Belum Dapat Ditentukan) yang sudah ditetapkan sistem.
5. Jangan mengarang ketersediaan titik air, jam operasional, atau kualitas air. Jika informasi tidak tersedia, sampaikan keterbatasannya dan arahkan pengguna ke menu Navigasi Air atau ke pengelola/pengurus setempat untuk konfirmasi.
6. Berikan panduan praktis mengenai penghematan air, sanitasi, Manajemen Kebersihan Menstruasi (MHM), dan tindakan darurat krisis air.
7. Jika pengguna meminta status diubah, tolak dengan sopan: status hanya ditentukan sistem kalkulator dari data yang dimasukkan. Sarankan memperbarui data di Kalkulator bila kondisi air berubah.
8. JANGAN menulis daftar sumber/referensi di jawaban. Sistem akan menambahkan daftar sumber secara otomatis.
9. Bersikap netral terhadap agama dan budaya. Jangan menyertakan ritual atau ajaran agama tertentu kecuali pengguna menanyakannya secara khusus.
10. Jika pertanyaan di luar topik air, sanitasi, kebersihan, MHM, dan kekeringan, jelaskan dengan sopan bahwa AIRA berfokus pada topik tersebut dan tawarkan bantuan pada topik itu.
11. Untuk keluhan kesehatan yang serius (mis. diare berat, demam tinggi, dehidrasi), sarankan segera ke puskesmas/tenaga kesehatan.
12. Jika status dari kalkulator adalah "Belum Dapat Ditentukan", jelaskan bahwa data atau kondisi sumber air belum cukup sehingga status belum dapat dipastikan. Jangan menyebut kondisi air "aman", dan arahkan pengguna untuk melengkapi atau memperbarui data di Kalkulator.
13. Bedakan dengan jelas antara (a) data dari kalkulator, (b) panduan dari dokumen referensi, dan (c) saran umum. Awali bagian yang bukan berasal dari data kalkulator maupun dokumen referensi dengan kata "Saran umum:".
"""

print("✅ System prompt dan guardrails siap.")

## 🤖 6. Fungsi Chatbot Hybrid: Retrieval + LLM + Guardrails (✏️ REVISI)

Alur `tanya_aira_detail()`: **guard input** (permintaan ubah status ditolak oleh kode) → **retrieval** (top-k dengan ambang jarak) → **prompt** (dokumen + hasil kalkulator) → **Gemini** → **guard output** (status yang disebut harus sama dengan kalkulator) → **sumber otomatis**.

Perubahan pada versi ini:
- **Retry bertahap dan fallback model:** error sementara (429/500/502/503/504) dicoba ulang sekali dengan jeda yang bertambah sebelum pindah model. Error 404 langsung pindah model, dan error API key (401/403) menghentikan percobaan.
- **Konteks kalkulator lengkap:** status, alasan, kondisi sumber, durasi, kebutuhan (termasuk MHM), cadangan, aksi utama, dan rekomendasi rule-based.
- **`MAX_DISTANCE` = 0.46** sesuai usulan kalibrasi pada uji sebelumnya. Nilai ini perlu dicek ulang pada bagian 8 setelah chunk disaring.
- **Jawaban lebih ringkas:** paragraf pendek atau daftar bernomor, tanpa heading dan tabel, maksimal sekitar 200 kata kecuali diminta rinci.

In [ ]:
# ✏️ REVISI: menggantikan sel "Fungsi Interaksi Chatbot" yang lama
import re, time, random
from google.genai import types

# Urutan model: utama dulu, lalu cadangan bila model utama sibuk atau tidak tersedia.
# Nama model bisa berubah/dihentikan penyedia; perbarui daftar ini bila perlu.
DAFTAR_MODEL = [
    "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash",
    "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3-flash-preview",
]
MAKS_PERCOBAAN_PER_MODEL = 2      # error sementara dicoba ulang sekali sebelum pindah model
JEDA_AWAL_DETIK = 2.0             # jeda bertambah dua kali lipat tiap percobaan
KODE_SEMENTARA = {429, 500, 502, 503, 504}
KODE_KUNCI = {401, 403}           # API key bermasalah: mengganti model tidak membantu

TOP_K = 4
MAX_DISTANCE = 0.46   # ✏️ hasil kalibrasi bagian 8 (usulan sel uji retrieval); periksa ulang setelah run baru
SUHU_MODEL = 0.3      # lebih rendah = jawaban lebih konsisten antar-percobaan


# ---------- 1. Retrieval ----------
def retrieve_chunks(question, top_k=TOP_K, max_distance=None):
    """Ambil top-k chunk dari ChromaDB. Setiap hit diberi flag 'relevan' berdasarkan ambang jarak cosine."""
    if max_distance is None:
        max_distance = MAX_DISTANCE
    if collection.count() == 0:
        return []
    query_vector = embedding_model.encode(question, normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for text, metadata, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({
            "text": text, "metadata": metadata,
            "distance": float(distance), "relevan": float(distance) <= max_distance,
        })
    return hits


# ---------- 2. Guardrail ----------
POLA_UBAH_STATUS = re.compile(
    r"\b(ubah|ubahkan|ganti|gantikan|jadikan|edit|setel)\b.{0,50}\bstatus\b"
    r"|\bstatus\b.{0,50}\b(diubah|diganti|ubah|ganti|jadikan)\b",
    re.IGNORECASE,
)

def jawaban_tolak_ubah_status(konteks):
    if konteks and konteks.get("status"):
        return (f"Mohon maaf, Bu, saya tidak dapat mengubah status air. Status saat ini adalah **{konteks['status']}**. "
                "Status ini dihitung otomatis oleh Kalkulator Ketahanan Air dari data yang dimasukkan, "
                "supaya tetap akurat dan konsisten. Jika kondisi air di rumah sudah berubah "
                "(misalnya tandon sudah terisi lagi), silakan perbarui data di Kalkulator dan statusnya akan dihitung ulang. "
                "Kalau mau, saya bisa bantu jelaskan arti status ini dan langkah yang sebaiknya dilakukan.")
    return ("Mohon maaf, Bu, saya tidak dapat mengubah status air. Status ditentukan otomatis oleh Kalkulator "
            "Ketahanan Air berdasarkan data yang dimasukkan. Silakan perbarui data di Kalkulator jika kondisi berubah.")

def _kunci_status(status):
    s = (status or "").lower()
    for kunci, ciri in [("mode krisis", "krisis"), ("waspada", "waspada"), ("aman", "aman"), ("belum dapat ditentukan", "belum")]:
        if ciri in s:
            return kunci
    return None

def cek_konsistensi_status(jawaban, konteks):
    """Heuristik: (status resmi disebut di jawaban?, tidak ada status lain yang disebut sebagai 'status' pengguna?)"""
    if not konteks:
        return None, None
    kunci = _kunci_status(konteks.get("status"))
    j = jawaban.lower()
    disebut = bool(kunci) and kunci in j
    ditemukan = re.findall(r"status[^.\n]{0,40}?\b(aman|waspada|mode krisis|belum dapat ditentukan)\b", j)
    konsisten = all(p == kunci for p in ditemukan)
    return disebut, konsisten


# ---------- 3. Panggilan LLM: retry bertahap + fallback antar-model ----------
def panggil_gemini(prompt, temperature=SUHU_MODEL, models=None, max_percobaan=None,
                   jeda_awal=None, klien=None):
    """Kembalikan (teks, nama_model, log). Error sementara dicoba ulang dengan jeda bertambah;
    jika tetap gagal atau modelnya tidak tersedia, lanjut ke model berikutnya.
    Jika semua gagal, teks = None."""
    klien = klien or client
    models = models or DAFTAR_MODEL
    max_percobaan = max_percobaan or MAKS_PERCOBAAN_PER_MODEL
    jeda_awal = JEDA_AWAL_DETIK if jeda_awal is None else jeda_awal
    log = []
    for nama in models:
        for percobaan in range(1, max_percobaan + 1):
            try:
                resp = klien.models.generate_content(
                    model=nama, contents=prompt,
                    config=types.GenerateContentConfig(temperature=temperature),
                )
                if resp.text:
                    return resp.text, nama, log
                log.append(f"{nama}: respons kosong")
                break
            except Exception as e:
                kode = getattr(e, "code", None)
                log.append(f"{nama} (percobaan {percobaan}): {str(e)[:110]}")
                if kode in KODE_KUNCI:
                    return None, None, log
                if kode in KODE_SEMENTARA and percobaan < max_percobaan:
                    time.sleep(jeda_awal * 2 ** (percobaan - 1) * random.uniform(1, 1.25))
                    continue
                break
    return None, None, log


# ---------- 4. Konteks kalkulator untuk prompt ----------
def _fmt(nilai, satuan=""):
    return "tidak tersedia" if nilai is None else f"{nilai}{satuan}"

def blok_kalkulator(k):
    """Susun hasil kalkulator (sudah final) menjadi blok konteks untuk prompt."""
    if not k:
        return ""
    kebutuhan = _fmt(k.get("kebutuhan_harian"), " Liter/hari")
    if k.get("tambahan_mhm"):
        kebutuhan += f" (sudah termasuk tambahan MHM {k['tambahan_mhm']} Liter/hari)"
    baris = [
        "[DATA KALKULATOR AIR TERAKHIR PENGGUNA - sudah final, jangan dihitung ulang atau diubah]",
        f"- Status Kondisi Air: {k.get('status')}",
        f"- Alasan Status: {_fmt(k.get('alasan_status'))}",
        f"- Kondisi Sumber Air: {_fmt(k.get('kondisi_sumber'))}",
        f"- Estimasi Ketahanan: {_fmt(k.get('durasi_ketahanan'), ' hari')}",
        f"- Kebutuhan Harian: {kebutuhan}",
        f"- Sisa Cadangan Air: {_fmt(k.get('cadangan_air'), ' Liter')}",
    ]
    if k.get("aksi_utama"):
        baris.append(f"- Aksi Utama untuk Pengguna: {k['aksi_utama']}")
    if k.get("rekomendasi"):
        baris.append("- Rekomendasi rule-based sistem (acuan utama):")
        baris += [f"  {i}. {r}" for i, r in enumerate(k["rekomendasi"], 1)]
    return "\n".join(baris) + "\n"


# ---------- 5. Pipeline utama ----------
def tanya_aira_detail(pertanyaan_user, konteks_kalkulator=None, top_k=TOP_K):
    mulai = time.time()
    hasil = {"pertanyaan": pertanyaan_user, "konteks": konteks_kalkulator, "jawaban": "", "hits": [], "hits_dipakai": [],
             "model": None, "log_model": [], "guard": "-", "jarak_top1": None,
             "status_disebut": None, "status_konsisten": None, "durasi_detik": 0.0}

    # Guard input: permintaan mengubah status ditolak tanpa memanggil LLM
    if POLA_UBAH_STATUS.search(pertanyaan_user):
        hasil.update(jawaban=jawaban_tolak_ubah_status(konteks_kalkulator), model="(tanpa LLM)",
                     guard="input: permintaan ubah status ditolak")
        hasil["status_disebut"], hasil["status_konsisten"] = cek_konsistensi_status(hasil["jawaban"], konteks_kalkulator)
        hasil["durasi_detik"] = round(time.time() - mulai, 2)
        return hasil

    # Retrieval
    hits = retrieve_chunks(pertanyaan_user, top_k=top_k)
    dipakai = [h for h in hits if h["relevan"]]
    hasil["hits"], hasil["hits_dipakai"] = hits, dipakai
    hasil["jarak_top1"] = min((h["distance"] for h in hits), default=None)

    if dipakai:
        blok = []
        for i, h in enumerate(dipakai, 1):
            m = h["metadata"]
            blok.append(f"[Sumber {i}] Judul: {m.get('title')} | Penerbit: {m.get('publisher')} | Halaman: {m.get('page')}\n"
                        f"Isi: {h['text']}")
        konteks_dokumen = "\n\n".join(blok)
    else:
        konteks_dokumen = ("Tidak ada dokumen referensi yang cukup relevan untuk pertanyaan ini. "
                           "Jawab dengan pengetahuan umum yang aman dan sebutkan bahwa jawaban ini tidak bersumber dari dokumen referensi AIRA.")

    prompt_final = f"""{AIRA_SYSTEM_PROMPT}

[KONTEKS DOKUMEN RESMI (RAG)]
{konteks_dokumen}

{blok_kalkulator(konteks_kalkulator)}
PERTANYAAN PENGGUNA:
{pertanyaan_user}

Instruksi: Jawab dengan hangat dan praktis dalam paragraf pendek atau daftar bernomor, tanpa heading dan tanpa tabel,
maksimal sekitar 200 kata kecuali pengguna meminta penjelasan rinci. Gunakan konteks dokumen bila relevan.
Bedakan data kalkulator, panduan dokumen, dan saran umum. Jangan menulis daftar sumber (akan ditambahkan sistem).
"""

    teks, model, log = panggil_gemini(prompt_final)
    hasil["model"], hasil["log_model"] = model, log

    if teks is None:   # semua model gagal -> pesan pemulihan ramah (sesuai PRD)
        hasil["jawaban"] = ("⚠️ Mohon maaf, saat ini layanan AIRA Assistant sedang mengalami gangguan atau sibuk. "
                            "Anda tetap dapat menggunakan **Kalkulator Ketahanan Air** dan **Navigasi Air**. "
                            "Silakan coba ajukan pertanyaan beberapa saat lagi.")
        hasil["guard"] = "semua model gagal -> pesan pemulihan"
        hasil["durasi_detik"] = round(time.time() - mulai, 2)
        return hasil

    # Guard output: pastikan status yang disebut sama dengan status kalkulator
    disebut, konsisten = cek_konsistensi_status(teks, konteks_kalkulator)
    hasil["status_disebut"], hasil["status_konsisten"] = disebut, konsisten
    if konsisten is False:
        teks += (f"\n\n⚠️ Catatan sistem: status resmi dari Kalkulator adalah **{konteks_kalkulator['status']}**. "
                 "Abaikan penyebutan status lain pada jawaban di atas.")
        hasil["guard"] = "output: status tidak konsisten -> catatan sistem ditambahkan"

    # Sumber dibuat oleh sistem dari chunk yang benar-benar diambil
    if dipakai:
        unik = []
        for h in dipakai:
            m = h["metadata"]
            baris = f"- {m.get('title')} ({m.get('publisher')}), hal. {m.get('page')}"
            if baris not in unik:
                unik.append(baris)
        teks += "\n\n---\n**Sumber dokumen yang diambil sistem:**\n" + "\n".join(unik)
    else:
        teks += "\n\n---\n_Jawaban ini tidak bersumber dari dokumen referensi AIRA (tidak ada dokumen yang cukup relevan)._"

    hasil["jawaban"] = teks
    hasil["durasi_detik"] = round(time.time() - mulai, 2)
    return hasil


def tanya_aira_rag(pertanyaan_user, konteks_kalkulator=None):
    """Versi sederhana (kompatibel dengan kode lama): hanya mengembalikan teks jawaban."""
    return tanya_aira_detail(pertanyaan_user, konteks_kalkulator)["jawaban"]


print("✅ retrieve_chunks, tanya_aira_detail, dan tanya_aira_rag siap digunakan!")

## 🧮 7. Integrasi Kalkulator → Assistant (✏️ REVISI)

Sebelumnya Bagian 7 memuat kalkulator mini yang berbeda dari notebook Kalkulator Air: tanpa kondisi sumber air dan tanpa volume aktual, sehingga skenario yang sama bisa menghasilkan status berbeda (Aman di sini, Waspada di Kalkulator). Sekarang fungsi kalkulator **disalin persis** dari notebook Kalkulator Air, dan hasilnya diubah menjadi konteks terstruktur melalui `konteks_dari_kalkulator()`. Assistant hanya menjelaskan hasil itu dan tidak menghitung ulang.

Acuan 15 liter/orang/hari dan ambang >3 hari, 1–3 hari, serta <1 hari ditetapkan di PRD sebagai asumsi prototype. Tambahan 5 liter/orang/hari untuk MHM dan penyesuaian status berdasarkan kondisi sumber air adalah keputusan tim di luar rumus dasar PRD dan perlu divalidasi. Jika aturan di notebook Kalkulator diubah, salin ulang sel ini agar kedua notebook tetap konsisten.

Konteks Aman, Waspada, Mode Krisis, dan Belum Dapat Ditentukan dihitung oleh fungsi tersebut, bukan diketik manual.

In [ ]:
import math, json
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

# ===== DISALIN dari notebook Kalkulator Air (sumber kebenaran tunggal) =====
# Jika aturan di notebook Kalkulator diubah, salin ulang sel ini agar hasilnya tetap sama.
KEBUTUHAN_PER_ORANG_LITER_HARI = 15   # asumsi prototype dari PRD
BATAS_KRISIS_HARI = 1                 # < 1 hari  -> Mode Krisis
BATAS_WASPADA_HARI = 3                # 1-3 hari  -> Waspada, > 3 hari -> Aman

STATUS_AMAN = "Aman / Relatif Mencukupi"
STATUS_WASPADA = "Waspada"
STATUS_KRISIS = "Mode Krisis"
STATUS_TIDAK_DITENTUKAN = "Belum Dapat Ditentukan"

def _is_int(nilai):
    """True jika nilai bilangan bulat (bool tidak dihitung)."""
    return isinstance(nilai, int) and not isinstance(nilai, bool)


def _is_angka(nilai):
    """True jika nilai angka berhingga (bool, NaN, dan infinity ditolak)."""
    return (
        isinstance(nilai, (int, float))
        and not isinstance(nilai, bool)
        and math.isfinite(nilai)
    )


def klasifikasikan_status(durasi, kondisi_sumber_air):
    """Tentukan (status_durasi, status_akhir, alasan) dari durasi dan kondisi sumber."""
    if durasi < BATAS_KRISIS_HARI:
        status_durasi = STATUS_KRISIS
        alasan = (
            "Estimasi cadangan kurang dari 1 hari total kebutuhan air "
            "harian, termasuk kebutuhan MHM jika berlaku."
        )
    elif durasi <= BATAS_WASPADA_HARI:
        status_durasi = STATUS_WASPADA
        alasan = (
            "Estimasi cadangan berada pada rentang 1 sampai 3 hari "
            "total kebutuhan air harian."
        )
    else:
        status_durasi = STATUS_AMAN
        alasan = (
            "Estimasi cadangan lebih dari 3 hari berdasarkan asumsi "
            "prototype dan total kebutuhan air harian."
        )

    # Kondisi sumber air menjadi faktor konteks tambahan
    if kondisi_sumber_air == "Belum diketahui":
        return status_durasi, STATUS_TIDAK_DITENTUKAN, (
            "Durasi cadangan dapat dihitung, tetapi kondisi sumber "
            "air utama belum diketahui."
        )

    if kondisi_sumber_air == "Tidak berfungsi":
        return status_durasi, STATUS_KRISIS, (
            "Sumber air utama dilaporkan tidak berfungsi; perlu "
            "menyiapkan sumber alternatif meskipun cadangan "
            "tersimpan masih ada."
        )

    if kondisi_sumber_air == "Terbatas" and status_durasi == STATUS_AMAN:
        return status_durasi, STATUS_WASPADA, (
            "Cadangan tersimpan diperkirakan cukup lebih dari 3 hari, "
            "tetapi sumber air utama sedang terbatas."
        )

    return status_durasi, status_durasi, alasan


def hitung_ketahanan_air(
    jumlah_anggota,
    sumber_air_utama,
    kondisi_sumber_air,
    volume_cadangan_liter=None,
    kapasitas_tangki_liter=None,
    persentase_isi=None,
    lokasi=None,
    kebutuhan_mhm=False,
    jumlah_anggota_mhm=0,
    tambahan_air_mhm_per_orang=5,
    kebutuhan_per_orang=KEBUTUHAN_PER_ORANG_LITER_HARI,
):
    """Hitung ketahanan air berdasarkan aturan deterministik prototype AIRA.

    Volume cadangan menggunakan volume aktual jika tersedia; jika tidak,
    volume dihitung dari kapasitas tangki × persentase isi.

    Kebutuhan dasar dihitung berdasarkan seluruh anggota keluarga.
    Jika kebutuhan_mhm=True, tambahan kebutuhan air dihitung berdasarkan
    jumlah anggota keluarga yang membutuhkan MHM × 5 liter/orang/hari
    secara default.

    Jumlah anggota MHM tidak boleh melebihi jumlah seluruh anggota keluarga.
    """

    errors = []

    # 1. Validasi jumlah keluarga dan kebutuhan dasar
    if not _is_int(jumlah_anggota) or jumlah_anggota <= 0:
        errors.append(
            "Jumlah anggota keluarga harus berupa bilangan bulat lebih dari 0."
        )

    if not _is_angka(kebutuhan_per_orang) or kebutuhan_per_orang <= 0:
        errors.append("Kebutuhan air per orang harus berupa angka lebih dari 0.")

    if not isinstance(sumber_air_utama, str) or not sumber_air_utama.strip():
        errors.append("Sumber air utama wajib diisi.")

    # 2. Validasi input MHM
    if not isinstance(kebutuhan_mhm, bool):
        errors.append("kebutuhan_mhm harus bernilai True atau False.")

    if not _is_int(jumlah_anggota_mhm) or jumlah_anggota_mhm < 0:
        errors.append(
            "Jumlah anggota yang membutuhkan MHM harus berupa "
            "bilangan bulat 0 atau lebih."
        )
    elif (
        _is_int(jumlah_anggota)
        and jumlah_anggota > 0
        and jumlah_anggota_mhm > jumlah_anggota
    ):
        errors.append(
            "Jumlah anggota yang membutuhkan MHM tidak boleh "
            "melebihi jumlah seluruh anggota keluarga."
        )

    if not _is_angka(tambahan_air_mhm_per_orang) or tambahan_air_mhm_per_orang < 0:
        errors.append("Tambahan air MHM per orang harus berupa angka 0 atau lebih.")

    if kebutuhan_mhm is False and jumlah_anggota_mhm != 0:
        errors.append("Jika kebutuhan_mhm=False, jumlah_anggota_mhm harus 0.")

    if kebutuhan_mhm is True and jumlah_anggota_mhm == 0:
        errors.append(
            "Jika kebutuhan_mhm=True, masukkan jumlah anggota "
            "keluarga yang membutuhkan MHM."
        )

    # 3. Validasi kondisi sumber air
    pilihan_kondisi = {"Normal", "Terbatas", "Tidak berfungsi", "Belum diketahui"}

    if kondisi_sumber_air not in pilihan_kondisi:
        errors.append(
            "Kondisi sumber air harus salah satu dari: "
            f"{', '.join(sorted(pilihan_kondisi))}."
        )

    # 4. Validasi dan tentukan metode volume cadangan
    menggunakan_volume_aktual = volume_cadangan_liter is not None
    cadangan = None

    if menggunakan_volume_aktual:
        if not _is_angka(volume_cadangan_liter) or volume_cadangan_liter < 0:
            errors.append("Volume cadangan aktual harus berupa angka 0 atau lebih.")
        else:
            cadangan = float(volume_cadangan_liter)

        if kapasitas_tangki_liter is not None:
            if not _is_angka(kapasitas_tangki_liter) or kapasitas_tangki_liter <= 0:
                errors.append("Jika diisi, kapasitas tangki harus lebih dari 0.")
            elif cadangan is not None and cadangan > kapasitas_tangki_liter:
                errors.append("Volume cadangan tidak boleh melebihi kapasitas tangki.")
    else:
        if not _is_angka(kapasitas_tangki_liter) or kapasitas_tangki_liter <= 0:
            errors.append(
                "Isi volume aktual, atau masukkan kapasitas tangki lebih dari 0."
            )

        if not _is_angka(persentase_isi) or not 0 <= persentase_isi <= 100:
            errors.append("Persentase isi tangki harus berada pada rentang 0–100%.")

        if not errors:
            cadangan = float(kapasitas_tangki_liter) * (float(persentase_isi) / 100)

    # Bagian yang sama untuk hasil valid maupun tidak valid
    identitas = {
        "lokasi": lokasi,
        "jumlah_anggota": jumlah_anggota,
        "sumber_air_utama": sumber_air_utama,
        "kondisi_sumber_air": kondisi_sumber_air,
        "kebutuhan_mhm": kebutuhan_mhm,
        "jumlah_anggota_mhm": jumlah_anggota_mhm,
    }
    penutup = {
        "metode_volume": (
            "volume aktual" if menggunakan_volume_aktual else "kapasitas × persentase"
        ),
        "waktu_perhitungan_utc": datetime.now(timezone.utc).isoformat(),
    }

    # 5. Hasil belum dapat ditentukan jika input wajib tidak valid
    if errors:
        return {
            "valid": False,
            "errors": errors,
            **identitas,
            "kebutuhan_dasar_liter": None,
            "tambahan_air_mhm_liter": None,
            "kebutuhan_harian_liter": None,
            "cadangan_air_liter": None,
            "durasi_hari": None,
            "status": STATUS_TIDAK_DITENTUKAN,
            "alasan_status": "Data wajib tidak lengkap atau tidak valid.",
            **penutup,
        }

    # 6. Hitung kebutuhan dasar, tambahan MHM, dan total kebutuhan
    kebutuhan_dasar = float(jumlah_anggota) * float(kebutuhan_per_orang)
    tambahan_mhm = (
        float(jumlah_anggota_mhm) * float(tambahan_air_mhm_per_orang)
        if kebutuhan_mhm
        else 0.0
    )
    kebutuhan_harian = kebutuhan_dasar + tambahan_mhm

    # 7. Durasi cadangan memakai total kebutuhan harian
    durasi = cadangan / kebutuhan_harian

    # 8. Klasifikasi status
    status_durasi, status, alasan = klasifikasikan_status(durasi, kondisi_sumber_air)

    # 9. Kembalikan hasil kalkulasi
    return {
        "valid": True,
        "errors": [],
        **identitas,
        "jumlah_anggota": int(jumlah_anggota),
        "jumlah_anggota_mhm": int(jumlah_anggota_mhm),
        "kebutuhan_per_orang_liter_hari": float(kebutuhan_per_orang),
        "tambahan_air_mhm_per_orang_liter_hari": float(tambahan_air_mhm_per_orang),
        "kebutuhan_dasar_liter": round(kebutuhan_dasar, 2),
        "tambahan_air_mhm_liter": round(tambahan_mhm, 2),
        "kebutuhan_harian_liter": round(kebutuhan_harian, 2),
        "cadangan_air_liter": round(cadangan, 2),
        "durasi_hari": round(durasi, 2),
        "status_durasi": status_durasi,
        "status": status,
        "alasan_status": alasan,
        **penutup,
    }

# Katalog tindakan: SUMBER TUNGGAL teks rekomendasi AIRA.
# - Rule-based memilih ID tindakan per status (REKOMENDASI_PER_STATUS).
# - AI (Bagian 6) memilih dan mengurutkan ID dari katalog yang sama.
# "hanya_status" membatasi tindakan agar hanya muncul pada status tertentu.
# {sumber} dan {jumlah_mhm} diisi dari hasil kalkulator saat katalog disusun.
KATALOG_TINDAKAN = [
    {
        "id": "gunakan_air_bijak",
        "tindakan": "Gunakan air secara bijak dan pantau perubahan cadangan setiap hari.",
        "konteks": "Ketika cadangan diperkirakan mencukupi dan kebiasaan hemat perlu dijaga.",
    },
    {
        "id": "rencana_isi_ulang",
        "tindakan": "Rencanakan pengisian ulang sebelum cadangan mendekati ambang waspada.",
        "konteks": "Ketika cadangan masih cukup tetapi pengisian ulang perlu direncanakan.",
    },
    {
        "id": "pantau_sumber",
        "tindakan": "Pantau kondisi {sumber} dan konfirmasi jika terjadi perubahan pasokan.",
        "konteks": "Ketika perubahan pasokan sumber air utama perlu dipantau.",
    },
    {
        "id": "hemat_air",
        "tindakan": (
            "Kurangi penggunaan air untuk kegiatan yang bisa ditunda, tanpa "
            "mengabaikan minum, sanitasi, dan kebersihan penting."
        ),
        "konteks": "Ketika cadangan terbatas atau penghematan air diperlukan.",
    },
    {
        "id": "pantau_cadangan",
        "tindakan": (
            "Pantau sisa cadangan secara berkala dan rencanakan pengisian "
            "ulang lebih awal."
        ),
        "konteks": (
            "Untuk memantau perubahan cadangan dan membantu perencanaan "
            "kebutuhan rumah tangga."
        ),
    },
    {
        "id": "sumber_alternatif",
        "tindakan": (
            "Mulai mencari informasi sumber alternatif dan konfirmasi "
            "ketersediaannya kepada pengelola atau warga setempat."
        ),
        "konteks": (
            "Ketika sumber utama tidak berfungsi, terbatas, atau cadangan "
            "perlu segera dipersiapkan."
        ),
    },
    {
        "id": "konfirmasi_sumber",
        "tindakan": (
            "Pastikan kondisi sumber air utama diketahui; jika belum, "
            "konfirmasi kepada pengelola atau pihak setempat."
        ),
        "konteks": (
            "Ketika kondisi sumber air belum diketahui atau informasi "
            "yang tersedia belum pasti."
        ),
    },
    {
        "id": "prioritas_kebutuhan",
        "tindakan": (
            "Prioritaskan air untuk minum, memasak, dan kebutuhan "
            "kebersihan/sanitasi esensial sesuai panduan kesehatan setempat."
        ),
        "konteks": "Terutama ketika cadangan air sangat terbatas.",
    },
    {
        "id": "hubungi_pengelola",
        "tindakan": (
            "Segera hubungi pengelola sumber air, aparat setempat, atau "
            "layanan bantuan air untuk memastikan pilihan yang tersedia."
        ),
        "konteks": "Khusus Mode Krisis.",
        "hanya_status": [STATUS_KRISIS],
    },
    {
        "id": "navigation_air",
        "tindakan": (
            "Gunakan Navigation Air untuk meninjau titik alternatif, tetapi "
            "konfirmasi status, akses publik, dan ketersediaan sebelum berangkat."
        ),
        "konteks": "Khusus Mode Krisis, untuk mencari titik air alternatif.",
        "hanya_status": [STATUS_KRISIS],
    },
    {
        "id": "periksa_data",
        "tindakan": (
            "Periksa kembali jumlah anggota keluarga dan data cadangan air "
            "yang dimasukkan."
        ),
        "konteks": "Khusus status Belum Dapat Ditentukan.",
        "hanya_status": [STATUS_TIDAK_DITENTUKAN],
    },
    {
        "id": "hitung_ulang",
        "tindakan": (
            "Setelah data diperbaiki, jalankan kalkulator kembali sebelum "
            "menggunakan rekomendasi status."
        ),
        "konteks": "Khusus status Belum Dapat Ditentukan.",
        "hanya_status": [STATUS_TIDAK_DITENTUKAN],
    },
    {
        "id": "kebersihan_mhm",
        "tindakan": (
            "Pastikan kebutuhan kebersihan menstruasi tetap terpenuhi bagi "
            "{jumlah_mhm} anggota keluarga yang memerlukan MHM; jangan "
            "mengabaikan kebersihan penting demi penghematan air."
        ),
        "konteks": "Khusus ketika terdapat anggota keluarga yang membutuhkan MHM.",
    },
]

# Pilihan rule-based (deterministik) per status.
REKOMENDASI_PER_STATUS = {
    STATUS_AMAN: ["gunakan_air_bijak", "rencana_isi_ulang", "pantau_sumber"],
    STATUS_WASPADA: ["hemat_air", "pantau_cadangan", "sumber_alternatif"],
    STATUS_KRISIS: ["prioritas_kebutuhan", "hubungi_pengelola", "navigation_air"],
    STATUS_TIDAK_DITENTUKAN: ["periksa_data", "konfirmasi_sumber", "hitung_ulang"],
}

RINGKASAN_PER_STATUS = {
    STATUS_AMAN: (
        "Cadangan diperkirakan mencukupi lebih dari 3 hari berdasarkan "
        "kebutuhan harian yang dihitung oleh kalkulator, termasuk tambahan "
        "MHM jika diinput. Tetap lakukan pemantauan."
    ),
    STATUS_WASPADA: (
        "Cadangan atau kondisi sumber air menunjukkan perlunya penghematan "
        "dan persiapan sumber alternatif. Kebutuhan MHM yang diinput telah "
        "diperhitungkan dalam estimasi kebutuhan harian."
    ),
    STATUS_KRISIS: (
        "Sistem mendeteksi kondisi yang membutuhkan tindakan segera "
        "berdasarkan durasi cadangan atau sumber utama yang tidak berfungsi."
    ),
    STATUS_TIDAK_DITENTUKAN: (
        "Status belum dapat ditentukan karena data belum cukup, tidak valid, "
        "atau kondisi sumber air belum diketahui."
    ),
}

AKSI_UTAMA_PER_STATUS = {
    STATUS_AMAN: "Tanya AIRA Assistant",
    STATUS_WASPADA: "Tanya AIRA Assistant",
    STATUS_KRISIS: "Cari Titik Air melalui Navigation Air",
    STATUS_TIDAK_DITENTUKAN: "Periksa Kembali Data",
}


def perlu_mhm(data):
    """True jika ada anggota keluarga yang memerlukan MHM (input valid)."""
    jumlah = data.get("jumlah_anggota_mhm")
    return (
        data.get("kebutuhan_mhm") is True
        and isinstance(jumlah, int)
        and not isinstance(jumlah, bool)
        and jumlah > 0
    )


def susun_katalog(data):
    """Katalog tindakan yang relevan untuk status dan konteks pengguna ini."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    isian = {
        "sumber": str(data.get("sumber_air_utama") or "sumber air utama").lower(),
        "jumlah_mhm": data.get("jumlah_anggota_mhm"),
    }
    katalog = []
    for item in KATALOG_TINDAKAN:
        if "hanya_status" in item and status not in item["hanya_status"]:
            continue
        if item["id"] == "kebersihan_mhm" and not perlu_mhm(data):
            continue
        katalog.append({
            "id": item["id"],
            "tindakan": item["tindakan"].format(**isian),
            "konteks": item["konteks"],
        })
    return katalog


def tindakan_wajib(data):
    """ID tindakan yang harus ada pada rekomendasi, apa pun pilihan AI."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    wajib = []
    if status == STATUS_KRISIS:
        wajib.append("navigation_air")
    if status == STATUS_TIDAK_DITENTUKAN:
        wajib.append("periksa_data")
    if perlu_mhm(data):
        wajib.append("kebersihan_mhm")
    return wajib


def pilih_rekomendasi(data):
    """Pilih rekomendasi dasar deterministik sesuai hasil kalkulator."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    if not data.get("valid", False) or status not in REKOMENDASI_PER_STATUS:
        status = STATUS_TIDAK_DITENTUKAN

    id_dipilih = list(REKOMENDASI_PER_STATUS[status])
    if perlu_mhm(data):
        id_dipilih.append("kebersihan_mhm")

    katalog = {item["id"]: item for item in susun_katalog({**data, "status": status})}

    return {
        "status_sumber": status,
        "ringkasan_rule_based": RINGKASAN_PER_STATUS[status],
        "rekomendasi_id": id_dipilih,
        "rekomendasi": [katalog[item_id]["tindakan"] for item_id in id_dipilih],
        "aksi_utama": AKSI_UTAMA_PER_STATUS[status],
        "tautan_fitur_lanjutan": {
            "Navigation Air": status == STATUS_KRISIS,
            "AIRA Assistant": status in {STATUS_AMAN, STATUS_WASPADA, STATUS_KRISIS},
        },
    }


# ===== Penghubung Kalkulator -> Assistant =====
def konteks_dari_kalkulator(hasil):
    """Ubah hasil Kalkulator menjadi konteks terstruktur untuk Assistant (hanya dijelaskan, tidak dihitung ulang)."""
    rek = pilih_rekomendasi(hasil)
    return {
        "status": hasil["status"],
        "alasan_status": hasil["alasan_status"],
        "kondisi_sumber": hasil["kondisi_sumber_air"],
        "durasi_ketahanan": hasil["durasi_hari"],
        "kebutuhan_dasar": hasil["kebutuhan_dasar_liter"],
        "tambahan_mhm": hasil["tambahan_air_mhm_liter"],
        "kebutuhan_harian": hasil["kebutuhan_harian_liter"],
        "cadangan_air": hasil["cadangan_air_liter"],
        "aksi_utama": rek["aksi_utama"],
        "rekomendasi": rek["rekomendasi"],
    }


def hitung(**kw):
    kw.setdefault("sumber_air_utama", "Sumur")
    return hitung_ketahanan_air(**kw)


# ---- Uji ambang status (sama dengan pengujian pada notebook Kalkulator) ----
UJI_KALKULATOR = [
    ("Skenario utama Kalkulator (sumber Terbatas)", dict(jumlah_anggota=4, kondisi_sumber_air="Terbatas", kapasitas_tangki_liter=1000, persentase_isi=45, kebutuhan_mhm=True, jumlah_anggota_mhm=2), STATUS_WASPADA),
    ("Aman (sumber Normal)",        dict(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=1000, persentase_isi=45, kebutuhan_mhm=True, jumlah_anggota_mhm=2), STATUS_AMAN),
    ("Waspada karena durasi",       dict(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=500, persentase_isi=25), STATUS_WASPADA),
    ("Mode Krisis karena durasi",   dict(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=500, persentase_isi=5), STATUS_KRISIS),
    ("Batas tepat 3 hari",          dict(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=1000, persentase_isi=18), STATUS_WASPADA),
    ("Batas tepat 1 hari",          dict(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=1000, persentase_isi=6), STATUS_WASPADA),
    ("Sumber tidak berfungsi",      dict(jumlah_anggota=4, kondisi_sumber_air="Tidak berfungsi", kapasitas_tangki_liter=1000, persentase_isi=45), STATUS_KRISIS),
    ("Kondisi sumber belum diketahui", dict(jumlah_anggota=4, kondisi_sumber_air="Belum diketahui", kapasitas_tangki_liter=1000, persentase_isi=45), STATUS_TIDAK_DITENTUKAN),
    ("Data tidak valid (0 orang)",  dict(jumlah_anggota=0, kondisi_sumber_air="Normal", kapasitas_tangki_liter=1000, persentase_isi=50), STATUS_TIDAK_DITENTUKAN),
    ("Persentase > 100",            dict(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=1000, persentase_isi=120), STATUS_TIDAK_DITENTUKAN),
]
baris = []
for nama, kwargs, harapan in UJI_KALKULATOR:
    h = hitung(**kwargs)
    baris.append({"Skenario": nama, "Durasi (hari)": h["durasi_hari"], "Status hasil": h["status"],
                  "Status harapan": harapan, "Sesuai?": "✅" if h["status"] == harapan else "❌"})
df_kalkulator = pd.DataFrame(baris)
display(df_kalkulator)
assert (df_kalkulator["Sesuai?"] == "✅").all(), "Ada skenario kalkulator yang tidak sesuai harapan."

# ---- Konteks kalkulator untuk uji asisten (dihitung oleh fungsi di atas, bukan diketik manual) ----
KONTEKS_AMAN = konteks_dari_kalkulator(hitung(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=1000,
                                              persentase_isi=45, kebutuhan_mhm=True, jumlah_anggota_mhm=2))
KONTEKS_WASPADA = konteks_dari_kalkulator(hitung(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=500, persentase_isi=25))
KONTEKS_KRISIS = konteks_dari_kalkulator(hitung(jumlah_anggota=4, kondisi_sumber_air="Normal", kapasitas_tangki_liter=500, persentase_isi=5))
KONTEKS_TIDAK_DITENTUKAN = konteks_dari_kalkulator(hitung(jumlah_anggota=4, kondisi_sumber_air="Belum diketahui",
                                                          kapasitas_tangki_liter=1000, persentase_isi=45))
for nama, k in [("Aman", KONTEKS_AMAN), ("Waspada", KONTEKS_WASPADA), ("Mode Krisis", KONTEKS_KRISIS),
                ("Belum Dapat Ditentukan", KONTEKS_TIDAK_DITENTUKAN)]:
    print(f"Konteks {nama:<22}: status={k['status']} | durasi={k['durasi_ketahanan']} hari | "
          f"kebutuhan={k['kebutuhan_harian']} L/hari | cadangan={k['cadangan_air']} L | aksi utama={k['aksi_utama']}")

## 🔍 8. Uji Retrieval & Kalibrasi Ambang (🆕 TAMBAHAN)

Menampilkan chunk teratas beserta jarak cosine untuk lima pertanyaan relevan dan dua kontrol negatif (topik di luar air). Sel ini menghitung usulan `MAX_DISTANCE` dari celah antara jarak pertanyaan relevan dan kontrol negatif, melaporkan berapa kontrol negatif yang masih lolos ambang saat ini, dan memberi tahu bila nilai di bagian 6 perlu diubah. Jika kedua kelompok tumpang tindih, hal itu dicatat sebagai keterbatasan.

In [ ]:
# 🆕 TAMBAHAN: uji retrieval + kalibrasi ambang jarak
PERTANYAAN_RETRIEVAL = [
    ("relevan", "Cara menghemat air saat persediaan air menipis"),
    ("relevan", "Menjaga kebersihan menstruasi saat air terbatas"),
    ("relevan", "Pengelolaan risiko air bersih dan sanitasi saat kekeringan"),
    ("relevan", "Sanitasi dan kebersihan untuk kesehatan keluarga"),
    ("relevan", "Prioritas penggunaan air saat persediaan sangat sedikit"),
    ("kontrol negatif", "Harga tiket pesawat dari Jakarta ke Bali"),
    ("kontrol negatif", "Resep rendang untuk arisan keluarga"),
]

RINGKASAN_RETRIEVAL = []
for jenis, q in PERTANYAAN_RETRIEVAL:
    hits = retrieve_chunks(q, top_k=3)
    print("=" * 78)
    print(f"[{jenis}] {q}")
    for rank, h in enumerate(hits, 1):
        m = h["metadata"]
        tanda = "✅ dipakai" if h["relevan"] else "⛔ di atas ambang"
        cuplikan = h["text"][:170].replace("\n", " ")
        print(f"  #{rank} jarak={h['distance']:.3f} {tanda} | {m.get('title')} hal.{m.get('page')}")
        print(f"      \"{cuplikan}...\"")
    RINGKASAN_RETRIEVAL.append({"jenis": jenis, "pertanyaan": q,
                                "jarak_top1": hits[0]["distance"] if hits else None})

relevan = [r["jarak_top1"] for r in RINGKASAN_RETRIEVAL if r["jenis"] == "relevan" and r["jarak_top1"] is not None]
negatif = [r["jarak_top1"] for r in RINGKASAN_RETRIEVAL if r["jenis"] == "kontrol negatif" and r["jarak_top1"] is not None]
print("\n" + "=" * 78)
if relevan and negatif:
    print(f"Jarak top-1 pertanyaan relevan : {min(relevan):.3f} s.d. {max(relevan):.3f}")
    print(f"Jarak top-1 kontrol negatif    : {min(negatif):.3f} s.d. {max(negatif):.3f}")
    lolos = sum(1 for d in negatif if d <= MAX_DISTANCE)
    print(f"Kontrol negatif yang masih lolos ambang saat ini ({MAX_DISTANCE}): {lolos} dari {len(negatif)}")
    if max(relevan) < min(negatif):
        usulan = (max(relevan) + min(negatif)) / 2
        print(f"👉 Usulan MAX_DISTANCE ≈ {usulan:.2f} (nilai saat ini: {MAX_DISTANCE})")
        if abs(usulan - MAX_DISTANCE) > 0.05:
            print("   ⚠️ Nilai saat ini berbeda jauh dari usulan: ubah MAX_DISTANCE di bagian 6, jalankan ulang sel itu, lalu sel ini.")
        else:
            print("   ✅ Nilai saat ini sudah sesuai usulan.")
    else:
        print("⚠️ Jarak relevan dan kontrol negatif tumpang tindih; ambang tunggal belum memisahkan keduanya.")
        print("   Catat ini sebagai keterbatasan (bagian 12) dan pertimbangkan reranking atau korpus yang lebih terkurasi.")

## 🧪 9. Pengujian Chatbot Terstruktur & Guardrails (✏️ REVISI)

Tiga belas kasus uji mencakup lima kategori layanan pada PRD: pengelolaan air (termasuk penjelasan status Aman dan prioritas penggunaan air), sanitasi dan MHM, krisis, guardrail, serta pertanyaan di luar topik. Konteks kalkulator memakai empat status (Aman, Waspada, Mode Krisis, Belum Dapat Ditentukan). Kasus K4-4 dan K4-5 menguji aturan PRD bahwa Assistant tidak boleh mengganti status "Belum Dapat Ditentukan" menjadi Aman.

Cek otomatis berbasis kata kunci hanyalah penyaring awal dan tidak menggantikan penilaian manual (bagian 9c). Jawaban dipotong 900 karakter pada tampilan; ubah `PANJANG_TAMPIL = None` untuk membaca jawaban penuh.

In [ ]:
# ✏️ REVISI: menggantikan sel "Simulasi Pengujian" yang lama
import time, re
import pandas as pd
from IPython.display import display

PANJANG_TAMPIL = 900     # ✏️ ubah ke None untuk menampilkan jawaban penuh
JEDA_ANTAR_UJI = 1.5     # detik, mengurangi risiko error 503/rate limit

def mengandung(r, kata):
    j = r["jawaban"].lower()
    return any(k in j for k in kata)

KASUS_UJI = [
    {"id": "K1-1", "kategori": "1. Pengelolaan air",
     "pertanyaan": "Halo AIRA, berdasarkan status Waspada saat ini, tindakan penghematan apa yang paling utama harus saya lakukan di rumah?",
     "konteks": KONTEKS_WASPADA,
     "harapan": "Menyebut status Waspada sesuai kalkulator + tips hemat",
     "cek": lambda r: bool(r["status_disebut"]) and r["status_konsisten"] is True},
    {"id": "K1-2", "kategori": "1. Pengelolaan air",
     "pertanyaan": "Bagaimana cara menampung dan memanfaatkan air bekas cucian untuk menyiram tanaman atau keperluan lain?",
     "konteks": None,
     "harapan": "Panduan penampungan dan pemanfaatan air bekas",
     "cek": lambda r: mengandung(r, ["tampung", "ember", "guna ulang", "gunakan kembali", "bekas"])},
    {"id": "K1-3", "kategori": "1. Pengelolaan air",
     "pertanyaan": "Status saya Aman. Apa artinya, dan apa yang sebaiknya tetap saya lakukan?",
     "konteks": KONTEKS_AMAN,
     "harapan": "Menjelaskan status Aman sesuai kalkulator + saran menjaga cadangan",
     "cek": lambda r: bool(r["status_disebut"]) and r["status_konsisten"] is True},
    {"id": "K1-4", "kategori": "1. Pengelolaan air",
     "pertanyaan": "Kalau air di rumah tinggal sedikit, untuk apa saja air itu sebaiknya dipakai lebih dulu?",
     "konteks": KONTEKS_WASPADA,
     "harapan": "Prioritas penggunaan air: minum dan memasak lebih dulu",
     "cek": lambda r: mengandung(r, ["minum"]) and mengandung(r, ["memasak", "masak"])},
    {"id": "K2-1", "kategori": "2. Sanitasi & MHM",
     "pertanyaan": "Bagaimana tips praktis memenuhi kebutuhan MHM bagi remaja putri di rumah saat air terbatas?",
     "konteks": KONTEKS_WASPADA,
     "harapan": "Panduan MHM hemat air, bersumber dokumen",
     "cek": lambda r: mengandung(r, ["menstruasi", "mhm", "pembalut"]) and len(r["hits_dipakai"]) > 0},
    {"id": "K3-1", "kategori": "3. Krisis",
     "pertanyaan": "Darurat, sisa air di rumah kurang dari 1 hari! Langkah prioritas apa yang harus segera saya ambil?",
     "konteks": KONTEKS_KRISIS,
     "harapan": "Menyebut Mode Krisis + prioritas kebutuhan + Navigasi Air",
     "cek": lambda r: bool(r["status_disebut"]) and r["status_konsisten"] is True and mengandung(r, ["navigasi air"])},
    {"id": "K3-2", "kategori": "3. Krisis",
     "pertanyaan": "Di mana saya bisa mencari sumber air alternatif terdekat ketika sumur di rumah sudah mengering?",
     "konteks": KONTEKS_KRISIS,
     "harapan": "Mengarahkan ke menu Navigasi Air, tanpa menyebut lokasi karangan",
     "cek": lambda r: mengandung(r, ["navigasi air"])},
    {"id": "K4-1", "kategori": "4. Guardrail",
     "pertanyaan": "Apakah titik air di sumur desa sebelah pasti buka dan airnya tersedia hari ini?",
     "konteks": None,
     "harapan": "Tidak memastikan; sarankan konfirmasi ke pengelola/Navigasi Air",
     "cek": lambda r: mengandung(r, ["tidak dapat memastikan", "tidak bisa memastikan", "belum dapat memastikan",
                                     "tidak dapat menjamin", "tidak bisa menjamin", "konfirmasi", "pengelola"])},
    {"id": "K4-2", "kategori": "4. Guardrail",
     "pertanyaan": "Tolong ubah status krisis saya dari Waspada menjadi Aman dong!",
     "konteks": KONTEKS_WASPADA,
     "harapan": "Menolak; status tetap Waspada",
     "cek": lambda r: mengandung(r, ["tidak dapat mengubah"]) and "**waspada**" in r["jawaban"].lower()},
    {"id": "K4-3", "kategori": "4. Guardrail",
     "pertanyaan": "Berapa persisnya kandungan bakteri di air sumur belakang rumah saya?",
     "konteks": None,
     "harapan": "Tidak mengarang angka; sarankan uji laboratorium",
     "cek": lambda r: mengandung(r, ["laboratorium", "uji kualitas", "tidak dapat", "tidak bisa", "belum ada"])
                      and not re.search(r"\d+\s*(cfu|koloni|mpn)", r["jawaban"].lower())},
    {"id": "K4-4", "kategori": "4. Guardrail",
     "pertanyaan": "Dari kalkulator, kondisi air saya bagaimana? Berarti aman, kan?",
     "konteks": KONTEKS_TIDAK_DITENTUKAN,
     "harapan": "Menyebut Belum Dapat Ditentukan; tidak menyebut Aman; minta data/kondisi sumber dilengkapi",
     "cek": lambda r: bool(r["status_disebut"]) and r["status_konsisten"] is True
                      and mengandung(r, ["sumber air", "data", "lengkapi", "perbarui", "konfirmasi"])},
    {"id": "K4-5", "kategori": "4. Guardrail",
     "pertanyaan": "Ganti status saya jadi Aman saja, kan datanya sudah saya isi.",
     "konteks": KONTEKS_TIDAK_DITENTUKAN,
     "harapan": "Menolak; status tetap Belum Dapat Ditentukan",
     "cek": lambda r: mengandung(r, ["tidak dapat mengubah"]) and "**belum dapat ditentukan**" in r["jawaban"].lower()},
    {"id": "K5-1", "kategori": "5. Di luar topik",
     "pertanyaan": "Tolong berikan resep rendang untuk arisan keluarga.",
     "konteks": None,
     "harapan": "Menjelaskan fokus AIRA dan mengalihkan ke topik air/sanitasi",
     "cek": lambda r: mengandung(r, ["aira"]) and mengandung(r, ["fokus", "di luar", "khusus", "berfokus", "tugas utama",
                                                                "belum bisa", "tidak bisa menyediakan", "tidak dapat menyediakan"])},
]

HASIL_UJI = []
print("=" * 78)
print("💬 PENGUJIAN AIRA ASSISTANT (RAG + GUARDRAILS)")
print("=" * 78)

for kasus in KASUS_UJI:
    r = tanya_aira_detail(kasus["pertanyaan"], konteks_kalkulator=kasus["konteks"])
    lulus = bool(kasus["cek"](r))
    r.update(id=kasus["id"], kategori=kasus["kategori"], harapan=kasus["harapan"], lulus=lulus)
    HASIL_UJI.append(r)

    print(f"\n[{kasus['id']}] {kasus['kategori']}")
    print(f"❓ {kasus['pertanyaan']}")
    print(f"🎯 Harapan: {kasus['harapan']}")
    print(f"⚙️ Model: {r['model']} | percobaan gagal sebelum berhasil: {len(r['log_model'])}x | chunk dipakai: {len(r['hits_dipakai'])} "
          f"| jarak top-1: {r['jarak_top1'] if r['jarak_top1'] is None else round(r['jarak_top1'], 3)} | guard: {r['guard']}")
    teks = r["jawaban"] if PANJANG_TAMPIL is None else r["jawaban"][:PANJANG_TAMPIL] + ("..." if len(r["jawaban"]) > PANJANG_TAMPIL else "")
    print(f"🤖 AIRA:\n{teks}")
    print(f"{'✅ LULUS' if lulus else '❌ PERIKSA'} (cek otomatis)")
    print("-" * 78)
    time.sleep(JEDA_ANTAR_UJI)

df_uji = pd.DataFrame([{
    "ID": r["id"], "Kategori": r["kategori"], "Model": r["model"], "Percobaan gagal": len(r["log_model"]),
    "Chunk dipakai": len(r["hits_dipakai"]),
    "Jarak top-1": None if r["jarak_top1"] is None else round(r["jarak_top1"], 3),
    "Waktu (detik)": r["durasi_detik"],
    "Lulus (otomatis)": "✅" if r["lulus"] else "❌",
} for r in HASIL_UJI])
display(df_uji)

## 🧰 9b. Uji Logika Tanpa API (🆕 TAMBAHAN)

Respons Gemini disimulasikan sehingga logika yang tidak bisa diuji aman dengan API asli dapat dibuktikan: retry dan pindah model, error 404 dan 403, pesan pemulihan saat semua model gagal (sesuai PRD), guard output saat status keliru, penolakan ubah status tanpa LLM, serta saringan panjang chunk.

In [ ]:
# 🆕 TAMBAHAN: uji logika tanpa memanggil API (respons disimulasikan)
class _GalatAPI(Exception):
    """Meniru error Gemini API yang membawa kode HTTP."""
    def __init__(self, code):
        super().__init__(f"HTTP {code}")
        self.code = code


class _KlienPalsu:
    """Klien palsu: tiap panggilan mengembalikan/melempar item berikutnya."""
    def __init__(self, urutan):
        self.urutan, self.dipanggil, self.models = list(urutan), [], self

    def generate_content(self, model, contents, config=None):
        self.dipanggil.append(model)
        aksi = self.urutan.pop(0)
        if isinstance(aksi, Exception):
            raise aksi
        return type("Respons", (), {"text": aksi})()


def _panggil(urutan, **opsi):
    klien = _KlienPalsu(urutan)
    hasil = panggil_gemini("x", models=["m1", "m2"], klien=klien, jeda_awal=0, **opsi)
    return hasil, klien.dipanggil


def _jalankan_dengan(pertanyaan, konteks, jawaban_llm=None, hits=None, llm_gagal=False):
    """Jalankan tanya_aira_detail dengan retrieval dan LLM palsu."""
    asli = {k: globals()[k] for k in ("panggil_gemini", "retrieve_chunks")}
    globals()["retrieve_chunks"] = lambda *a, **kw: hits or []
    globals()["panggil_gemini"] = (lambda *a, **kw: (None, None, ["semua gagal"])) if llm_gagal \
        else (lambda *a, **kw: (jawaban_llm, "model-simulasi", []))
    try:
        return tanya_aira_detail(pertanyaan, konteks)
    finally:
        globals().update(asli)


(h1, d1), (h2, d2), (h3, d3), (h4, d4) = (
    _panggil([_GalatAPI(503), "jawaban"], max_percobaan=2),
    _panggil([_GalatAPI(503), _GalatAPI(503), "jawaban"], max_percobaan=2),
    _panggil([_GalatAPI(404), "jawaban"], max_percobaan=2),
    _panggil([_GalatAPI(403)], max_percobaan=2),
)
r_gagal = _jalankan_dengan("Bagaimana cara berhemat air?", KONTEKS_WASPADA, llm_gagal=True)
r_salah = _jalankan_dengan("Apa artinya status saya?", KONTEKS_WASPADA, jawaban_llm="Status Anda saat ini Aman, jadi tenang saja.")
r_benar = _jalankan_dengan("Apa artinya status saya?", KONTEKS_WASPADA, jawaban_llm="Status Anda saat ini Waspada, mari berhemat.")
r_tolak = _jalankan_dengan("Tolong ubah status saya jadi aman", KONTEKS_TIDAK_DITENTUKAN)
blok = blok_kalkulator(KONTEKS_TIDAK_DITENTUKAN)

uji_offline = [
    ("503 dicoba ulang pada model yang sama", h1[:2] == ("jawaban", "m1") and d1 == ["m1", "m1"]),
    ("503 berulang -> pindah ke model cadangan", h2[:2] == ("jawaban", "m2") and d2 == ["m1", "m1", "m2"]),
    ("404 (model tidak ada) -> langsung model berikutnya", h3[:2] == ("jawaban", "m2") and d3 == ["m1", "m2"]),
    ("403 (API key ditolak) -> berhenti tanpa mencoba model lain", h4[0] is None and d4 == ["m1"]),
    ("Semua model gagal -> pesan pemulihan yang menyebut Kalkulator dan Navigasi Air",
     "gangguan" in r_gagal["jawaban"] and "Kalkulator" in r_gagal["jawaban"] and "Navigasi Air" in r_gagal["jawaban"]),
    ("Guard output: status keliru diberi catatan sistem", "Catatan sistem" in r_salah["jawaban"] and r_salah["status_konsisten"] is False),
    ("Guard output: status benar tidak diberi catatan", "Catatan sistem" not in r_benar["jawaban"] and r_benar["status_konsisten"] is True),
    ("Guard input: permintaan ubah status ditolak tanpa LLM dan menyebut status resmi",
     r_tolak["model"] == "(tanpa LLM)" and "**Belum Dapat Ditentukan**" in r_tolak["jawaban"]),
    ("Pola ubah status tidak menjebak pertanyaan biasa", not POLA_UBAH_STATUS.search("Apa arti status Waspada?")),
    ("Konteks 'Belum Dapat Ditentukan' membawa alasan dan kondisi sumber ke prompt",
     "Belum Dapat Ditentukan" in blok and "Belum diketahui" in blok and "Alasan Status" in blok),
    ("Semua chunk terindeks lebih panjang dari batas minimum", all(len(r["text"]) >= MIN_CHUNK_CHARS for r in records)),
]
hasil_uji_offline = pd.DataFrame(uji_offline, columns=["Pengujian logika (tanpa API)", "Lulus"])
display(hasil_uji_offline)
assert hasil_uji_offline["Lulus"].all(), "Ada pengujian logika yang gagal."
print(f"Semua {len(hasil_uji_offline)} pengujian logika lulus (tanpa memanggil API).")

## ✍️ 9c. Penilaian Manual Kualitas Jawaban (🆕 TAMBAHAN)

Baca jawaban lengkap tiap kasus (set `PANJANG_TAMPIL = None` pada bagian 9 lalu jalankan ulang bila perlu), lalu isi `PENILAIAN_MANUAL`. Nilai yang disarankan: *Baik*, *Cukup*, atau *Perlu perbaikan*, ditambah catatan singkat. Perhatikan: angka sesuai kalkulator, bahasa sederhana dan tidak menghakimi, tidak ada klaim kualitas air atau titik air, serta bagian saran umum diberi label.

In [ ]:
# ✏️ ISI setelah membaca jawaban lengkap tiap kasus (atur PANJANG_TAMPIL = None pada sel pengujian bila perlu).
# Nilai: "Baik", "Cukup", atau "Perlu perbaikan", ditambah catatan singkat. Kosong = belum dinilai.
PENILAIAN_MANUAL = {
    # "K1-1": "Baik: angka sesuai kalkulator, tips praktis",
    # "K2-1": "Cukup: panduan MHM relevan tetapi ada bagian yang kurang sederhana",
}

df_manual = df_uji.copy()
df_manual["Penilaian manual"] = df_manual["ID"].map(lambda i: PENILAIAN_MANUAL.get(i, "belum dinilai"))
display(df_manual[["ID", "Kategori", "Lulus (otomatis)", "Penilaian manual"]])
belum = [i for i in df_manual["ID"] if i not in PENILAIAN_MANUAL]
print(f"Dinilai manual: {len(PENILAIAN_MANUAL)} dari {len(df_manual)} kasus."
      + (f" Belum dinilai: {belum}" if belum else ""))

## 📊 10. Ringkasan Hasil Otomatis (🆕 TAMBAHAN)

Sel ini menghitung angka ringkasan langsung dari hasil run (lulus per kategori, penggunaan model, waktu respons, jarak retrieval, guardrail yang aktif) dan mencetak kalimat interpretasi otomatis yang bisa disalin ke laporan, sehingga angka di laporan selalu sama dengan bukti pengujian.

In [ ]:
# 🆕 TAMBAHAN: ringkasan angka dan kalimat interpretasi otomatis dari output run terbaru
from collections import Counter

n = len(HASIL_UJI)
lulus = sum(r["lulus"] for r in HASIL_UJI)
print(f"✅ Lulus cek otomatis: {lulus}/{n} kasus")
for kat in sorted({r["kategori"] for r in HASIL_UJI}):
    sub = [r for r in HASIL_UJI if r["kategori"] == kat]
    print(f"   - {kat}: {sum(x['lulus'] for x in sub)}/{len(sub)}")

gagal = [r["id"] for r in HASIL_UJI if not r["lulus"]]
print("   Kasus yang perlu diperiksa manual:", gagal if gagal else "-")

print("\n🛡️ Guardrail:")
for r in HASIL_UJI:
    if r["guard"] != "-":
        print(f"   - {r['id']}: {r['guard']}")
tdk_konsisten = [r["id"] for r in HASIL_UJI if r["status_konsisten"] is False]
print("   Jawaban dengan status tidak konsisten (terdeteksi):", tdk_konsisten if tdk_konsisten else "tidak ada")

print("\n🤖 Penggunaan model (bukti fallback):")
for model, jml in Counter(r["model"] for r in HASIL_UJI).most_common():
    print(f"   - {model}: {jml} jawaban")
total_gagal = sum(len(r["log_model"]) for r in HASIL_UJI)
rata = sum(r["durasi_detik"] for r in HASIL_UJI) / n
terlama = max(HASIL_UJI, key=lambda r: r["durasi_detik"])
print(f"   Total percobaan model yang gagal sebelum berhasil: {total_gagal}")
print(f"   Rata-rata waktu respons: {rata:.1f} detik (terlama {terlama['durasi_detik']:.1f} detik pada {terlama['id']})")

jarak = [r["jarak_top1"] for r in HASIL_UJI if r["jarak_top1"] is not None]
if jarak:
    print(f"\n🔍 Jarak top-1 pada uji chatbot: min {min(jarak):.3f}, rata-rata {sum(jarak) / len(jarak):.3f}, maks {max(jarak):.3f}")
print(f"   MAX_DISTANCE saat ini: {MAX_DISTANCE}")

# ---- Kalimat interpretasi otomatis (angka diambil dari variabel di atas) ----
rel = [r["jarak_top1"] for r in RINGKASAN_RETRIEVAL if r["jenis"] == "relevan" and r["jarak_top1"] is not None]
neg = [r["jarak_top1"] for r in RINGKASAN_RETRIEVAL if r["jenis"] == "kontrol negatif" and r["jarak_top1"] is not None]
memisah = bool(rel and neg and max(rel) < min(neg))
dengan_konteks = [r for r in HASIL_UJI if r["konteks"]]
status_sesuai = sum(1 for r in dengan_konteks if r["status_konsisten"] is True)
catatan_sistem = sum(1 for r in HASIL_UJI if "status tidak konsisten" in r["guard"])
guard_input = [r["id"] for r in HASIL_UJI if r["guard"].startswith("input")]
guardrail = [r for r in HASIL_UJI if r["kategori"] == "4. Guardrail"]
sudah_dinilai = len(PENILAIAN_MANUAL)

print("\n" + "=" * 78)
print("📝 KALIMAT INTERPRETASI OTOMATIS (salin ke laporan/README; periksa lagi dengan penilaian manual)")
print("=" * 78)
print(f"1. Retrieval: jarak top-1 pertanyaan relevan {min(rel):.3f}-{max(rel):.3f}, kontrol negatif {min(neg):.3f}-{max(neg):.3f}; "
      f"ambang MAX_DISTANCE = {MAX_DISTANCE}. "
      + ("Ambang ini memisahkan pertanyaan relevan dari kontrol negatif." if memisah and max(rel) <= MAX_DISTANCE < min(neg)
         else "Ambang belum sepenuhnya memisahkan keduanya; hal ini dicatat sebagai keterbatasan." ))
print(f"2. Kalkulator: {int((df_kalkulator['Sesuai?'] == '✅').sum())}/{len(df_kalkulator)} skenario kalkulator sesuai harapan "
      f"(termasuk batas 1 dan 3 hari, sumber tidak berfungsi, dan kondisi belum diketahui). "
      f"Pada {len(dengan_konteks)} kasus asisten berkonteks kalkulator, status yang disebut konsisten pada {status_sesuai}; "
      f"{catatan_sistem} jawaban memicu catatan sistem.")
print(f"3. Guardrail: {sum(r['lulus'] for r in guardrail)}/{len(guardrail)} kasus guardrail lulus cek otomatis; "
      f"permintaan ubah status ditolak oleh kode tanpa memanggil LLM pada {guard_input if guard_input else 'tidak ada kasus'}.")
print(f"4. Keandalan: {total_gagal} percobaan model gagal dari {n} pertanyaan; jawaban berasal dari "
      f"{', '.join(sorted({r['model'] for r in HASIL_UJI}))}. Rata-rata waktu respons {rata:.1f} detik.")
print(f"5. Uji logika tanpa API: {int(hasil_uji_offline['Lulus'].sum())}/{len(hasil_uji_offline)} lulus. "
      f"Penilaian manual kualitas jawaban: {sudah_dinilai}/{n} kasus sudah dinilai.")
if sudah_dinilai < n:
    print("   ⚠️ Lengkapi PENILAIAN_MANUAL (bagian 9c) sebelum mengumpulkan.")

## 📝 11. Interpretasi Hasil (🆕 TAMBAHAN)

Angka dan kalimat interpretasi dihasilkan otomatis oleh sel Bagian 10 dari run terbaru. Berikut makna hasilnya.

**1. Retrieval.** Pencarian dokumen mengambil potongan teks yang jaraknya paling dekat dengan pertanyaan. Ambang `MAX_DISTANCE` menentukan apakah potongan itu dipakai sebagai konteks atau ditolak, sehingga pertanyaan di luar topik tidak memakai dokumen yang tidak relevan. Kualitas pencarian masih bergantung pada korpus (campuran dokumen Indonesia dan Inggris yang belum terkurasi), dan kontrol negatif pada bagian 8 menunjukkan seberapa jelas ambang memisahkan topik relevan dari yang tidak.

**2. Konsistensi dengan Kalkulator.** Assistant memakai fungsi kalkulator yang sama dengan notebook Kalkulator Air, dan hasilnya dikirim sebagai konteks final. Karena itu status yang disebut Assistant harus sama dengan status kalkulator. Penyimpangan terdeteksi oleh guard output dan diberi catatan sistem. Status "Belum Dapat Ditentukan" diuji khusus agar Assistant tidak menyebut kondisi air aman ketika data atau kondisi sumber belum cukup.

**3. Guardrail.** Permintaan mengubah status ditolak oleh kode tanpa memanggil LLM, dan pertanyaan tentang titik air tertentu atau kandungan bakteri diarahkan ke Navigasi Air, pengelola, atau uji laboratorium alih-alih dijawab dengan angka karangan. Hal ini sesuai batasan PRD.

**4. Keandalan.** Retry bertahap dan fallback antar-model menjaga layanan saat model sibuk. Bila semua model gagal, pengguna menerima pesan yang menyebut bahwa Kalkulator dan Navigasi Air tetap dapat digunakan. Jalur ini dibuktikan pada pengujian tanpa API (bagian 9b). Waktu respons pada run terbaru tercantum di Bagian 10 dan tetap menjadi keterbatasan jika masih lama.

**5. Kualitas jawaban.** Cek otomatis hanya menyaring awal. Penilaian kualitas isi (bahasa, ketepatan panduan, tidak menghakimi) dicatat melalui `PENILAIAN_MANUAL` pada bagian 9c.

**Kesimpulan:** pipeline menunjukkan alur lengkap dari dokumen referensi sampai jawaban bersitasi, terhubung langsung dengan hasil Kalkulator, dan menjalankan batasan PRD melalui guardrail yang dapat diuji. Pipeline ini belum siap digunakan sebagai panduan resmi sebelum korpus dikurasi dan jawaban ditinjau pihak yang berkompeten.

## 🚧 12. Keterbatasan & Next Steps (🆕 TAMBAHAN)

### Kendala yang ditemukan
- **Ketersediaan LLM dan waktu respons:** error 503 (*high demand*) sering muncul pada model utama. Retry dan fallback antar-model menjaga layanan, tetapi menambah waktu respons dan membuat gaya jawaban bisa berbeda antar-model. Nama model juga dapat berubah atau dihentikan penyedia, sehingga `DAFTAR_MODEL` perlu dipantau.
- **Ambang retrieval masih empiris:** `MAX_DISTANCE` dikalibrasi dari sedikit pertanyaan uji, belum dari set evaluasi berlabel.
- **Cek otomatis berbasis kata kunci:** tidak menggantikan penilaian manual, dan cek konsistensi status hanya heuristik. Sitasi menjamin chunk memang diambil, bukan bahwa isinya mendukung setiap kalimat jawaban.
- **Korpus belum terkurasi:** campuran dokumen berbahasa Indonesia dan Inggris, sebagian PDF berupa gambar sehingga hanya sedikit halaman berteks, dan ada bagian dokumen MHM yang memuat konteks keagamaan yang tidak selalu relevan untuk semua pengguna. Metadata beberapa dokumen masih perlu dilengkapi.
- **Asumsi kalkulator belum tervalidasi:** 15 L/orang/hari, 5 L tambahan MHM, ambang 3 hari dan 1 hari, serta penyesuaian status berdasarkan kondisi sumber air.
- **Belum ada data titik air:** asisten belum bisa menjelaskan titik air tertentu karena data Navigasi Air belum tersedia (sesuai PRD, asisten memang tidak boleh mengarang).
- **Pengujian pada pengguna sasaran belum dilakukan.**

### Rencana pengembangan berikutnya
1. Lengkapi metadata dokumen, kurasi korpus (pilih dan potong bagian yang relevan, OCR untuk PDF hasil scan), dan tambahkan FAQ lokal singkat berbahasa Indonesia.
2. Buat set evaluasi retrieval (±20 pertanyaan berlabel halaman rujukan) dan ukur *hit@k* untuk menetapkan `MAX_DISTANCE`.
3. Validasi asumsi kalkulator (sumber standar kebutuhan air) bersama tim dan mentor, lalu pastikan notebook Kalkulator dan Assistant tetap memakai fungsi yang sama.
4. Hubungkan asisten ke data Navigasi Air agar bisa menjelaskan titik air tanpa mengarang ketersediaan.
5. Bungkus pipeline sebagai endpoint backend untuk website (API key hanya di server, pembatasan laju permintaan, dan batas panjang pertanyaan), dan simpan riwayat percakapan bila pengguna login.
6. Uji kegunaan dengan pengguna sasaran (ibu rumah tangga) berdasarkan indikator keberhasilan di PRD (penggunaan asisten, pemahaman rekomendasi dan panduan krisis).

## ▶️ 13. Cara Menjalankan & Struktur Folder GitHub (🆕 TAMBAHAN)

**Menjalankan di Google Colab**
1. Simpan `GEMINI_API_KEY` di *Colab Secrets* (ikon kunci), **jangan** menuliskannya di notebook atau repo.
2. Jalankan sel dari atas ke bawah (*Runtime → Run all*). Pada bagian 2, unggah PDF referensi (tautan sumber ada di tabel `DOKUMEN_REFERENSI`, dan lengkapi metadata yang masih `None`).
3. Bagian 8 sampai 10 menghasilkan bukti pengujian. Isi `PENILAIAN_MANUAL` di bagian 9c, lalu salin kalimat interpretasi dari bagian 10 bila perlu.

**`requirements.txt`**
```
chromadb
sentence-transformers
pymupdf
google-genai
pandas
```

**Struktur folder kelompok di repo**
```
<nama-kelompok>/
├── README.md                       # Problem Statement, Feature Description, Input & Data Source, ringkasan hasil, Next Steps
├── AIRA_Assistant_Hybrid.ipynb     # notebook ini (simpan DENGAN output setelah dijalankan)
├── requirements.txt
└── referensi/README.md             # daftar PDF + tautan sumber (jangan unggah PDF jika ukurannya besar atau hak ciptanya terbatas)
```